# Case Study: Titanic - Identificar MCAR, MAR e MNAR com MissingFCUP

Este notebook testa, de forma visual e didatica, se as visualizacoes do package ajudam a identificar **MCAR**, **MAR** e **MNAR**.

Vamos trabalhar sempre com duas versoes do dataset:
- **Original completo** (sem missing values)
- **Versao com missingness** gerada com `pyampute`

**Legenda de cores** (consistente em todas as figuras):
- Verde = valor presente
- Vermelho = valor em falta

## 1. Setup

In [1]:
# Se estiveres num ambiente limpo, podes instalar dependencias:
# %pip install -q seaborn pyampute

import numpy as np
import pandas as pd
import seaborn as sns

from pyampute.ampute import MultivariateAmputation

from missingfcup.core.MissingData import MissingData
from missingfcup.plots.Panel import Panel

## 2. Dataset (Titanic via seaborn)

Carregamos o dataset e removemos os NaNs originais para criar um **ground truth completo**.
Criamos duas variaveis derivadas simples para a analise:
- `family_size` = sibsp + parch + 1
- `fare_per_person` = fare / family_size

In [2]:
# Nota: seaborn.load_dataset faz download do dataset
# Se estiveres offline, precisas de ter o CSV localmente

_df_raw = sns.load_dataset("titanic")

# Ground truth completo
df_complete = _df_raw.dropna().reset_index(drop=True)

# Variaveis derivadas (continuas e intuitivas)
df_complete["family_size"] = df_complete["sibsp"] + df_complete["parch"] + 1
df_complete["fare_per_person"] = df_complete["fare"] / df_complete["family_size"]

print(df_complete.shape)
df_complete.head()

(182, 17)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,fare_per_person
0,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2,35.641650
1,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,2,26.550000
2,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True,1,51.862500
3,1,3,female,4.0,1,1,16.7000,S,Third,child,False,G,Southampton,yes,False,3,5.566667
4,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True,1,26.550000


## 3. Funcoes auxiliares (pyampute + comparacao)

Usamos `pyampute` para gerar missingness controlada:
- **MCAR**: missingness aleatoria
- **MAR**: missingness depende de uma variavel observada
- **MNAR**: missingness depende do proprio valor da variavel

A funcao abaixo amputa apenas a coluna alvo e devolve um novo DataFrame.

In [ ]:
from typing import Optional

PRESENT_COLOR = "#2ca02c"
MISSING_COLOR = "#d62728"
RATE_COLORSCALE = [[0.0, PRESENT_COLOR], [1.0, MISSING_COLOR]]

numeric_cols = [
    "age",
    "fare",
    "sibsp",
    "parch",
    "pclass",
    "fare_per_person",
]


def ampute_with_pyampute(
    df: pd.DataFrame,
    target_col: str,
    mechanism: str,
    driver_col: Optional[str] = None,
    prop: float = 0.30,
    seed: int = 42,
    score_func: str = "SIGMOID-RIGHT",
) -> pd.DataFrame:
    numeric_df = df[numeric_cols].copy()

    required = [target_col] + ([driver_col] if driver_col else [])
    for col in required:
        if col not in numeric_df.columns:
            raise ValueError(f"Column not found in numeric data: {col}")

    if mechanism == "MCAR":
        patterns = [
            {
                "incomplete_vars": [target_col],
                "mechanism": "MCAR",
            }
        ]
    elif mechanism == "MAR":
        if driver_col is None:
            raise ValueError("driver_col is required for MAR")
        patterns = [
            {
                "incomplete_vars": [target_col],
                "mechanism": "MAR",
                "weights": {driver_col: 1.0},
                "score_to_probability_func": score_func,
            }
        ]
    elif mechanism == "MNAR":
        patterns = [
            {
                "incomplete_vars": [target_col],
                "mechanism": "MNAR",
                "weights": {target_col: 1.0},
                "score_to_probability_func": score_func,
            }
        ]
    else:
        raise ValueError(f"Unknown mechanism: {mechanism}")

    amputer = MultivariateAmputation(prop=prop, patterns=patterns, seed=seed)
    amputed_numeric = amputer.fit_transform(numeric_df)

    df_out = df.copy()
    df_out[target_col] = amputed_numeric[target_col]
    return df_out


def compare_panel(title: str, left_plot, right_plot):
    panel = Panel(
        plots=[left_plot, right_plot],
        title=title,
        max_cols=2,
    )
    panel.show()


md_original = MissingData(df_complete)

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

# 4. MCAR - Missing Completely At Random

**Objetivo**: 30% de missingness em `age`, sem qualquer padrao.

O que esperamos ver:
- Pontos vermelhos espalhados sem concentracao em grupos especificos
- Nenhuma relacao clara com outras variaveis

Em cada visualizacao, a esquerda esta o dataset **original**, e a direita o dataset **com MCAR**.

In [ ]:
MCAR_TARGET = "age"

df_mcar = ampute_with_pyampute(
    df_complete,
    target_col=MCAR_TARGET,
    mechanism="MCAR",
    prop=0.30,
    seed=7,
)

md_mcar = MissingData(df_mcar)

### MCAR - Visualizacao 1 (nao diz o mecanismo)

O grafico de contagem mostra **quanto** falta, mas nao **por que** falta.

In [ ]:
compare_panel(
    "MCAR: Missing count por coluna (inconclusivo)",
    md_original.missing_count_barchart(
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mcar.missing_count_barchart(
        title="Com MCAR (30% em age)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MCAR - Visualizacao 2 (nao diz o mecanismo)

A taxa por coluna confirma a magnitude, mas nao revela padrao causal.

In [ ]:
compare_panel(
    "MCAR: Missing rate por coluna (inconclusivo)",
    md_original.column_missing_rate_heatmap(
        title="Original (completo)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
    md_mcar.column_missing_rate_heatmap(
        title="Com MCAR (30% em age)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
)

### MCAR - Visualizacao 3 (ajuda a identificar)

No heatmap, o vermelho aparece **espalhado**, sem concentracao por grupos.

In [ ]:
cols_mcar = ["age", "fare", "pclass", "sibsp", "parch"]

compare_panel(
    "MCAR: Heatmap de missingness (padrao aleatorio)",
    md_original.heatmap(
        selected_columns=cols_mcar,
        title="Original (sem missing)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mcar.heatmap(
        selected_columns=cols_mcar,
        title="MCAR em age",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MCAR - Visualizacao 4 (ajuda a identificar)

No scatterplot `age` vs `fare`, os pontos em falta aparecem dispersos,
sem concentracao em faixas especificas de `fare`.

In [ ]:
compare_panel(
    "MCAR: Scatterplot age vs fare (sem padrao)",
    md_original.scatterplot(
        x="age",
        y="fare",
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mcar.scatterplot(
        x="age",
        y="fare",
        title="Com MCAR (age)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

# 5. MAR - Missing At Random

**Objetivo**: 30% de missingness em `fare`, dependendo de `pclass`.

O que esperamos ver:
- Missingness concentrada em certos valores de `pclass`
- Um padrao visivel quando ordenamos ou cruzamos com a variavel observada

Em cada visualizacao, a esquerda esta o dataset **original**, e a direita o dataset **com MAR**.

In [ ]:
MAR_TARGET = "fare"
MAR_DRIVER = "pclass"

df_mar = ampute_with_pyampute(
    df_complete,
    target_col=MAR_TARGET,
    mechanism="MAR",
    driver_col=MAR_DRIVER,
    prop=0.30,
    seed=11,
)

md_mar = MissingData(df_mar)

### MAR - Visualizacao 1 (nao diz o mecanismo)

In [ ]:
compare_panel(
    "MAR: Missing count por coluna (inconclusivo)",
    md_original.missing_count_barchart(
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.missing_count_barchart(
        title="Com MAR (30% em fare)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MAR - Visualizacao 2 (nao diz o mecanismo)

In [ ]:
compare_panel(
    "MAR: Missing rate por coluna (inconclusivo)",
    md_original.column_missing_rate_heatmap(
        title="Original (completo)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
    md_mar.column_missing_rate_heatmap(
        title="Com MAR (30% em fare)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
)

### MAR - Visualizacao 3 (ajuda a identificar)

Ordenando por `pclass`, o missingness em `fare` aparece concentrado em certas classes.

In [ ]:
cols_mar = ["pclass", "fare", "age", "sibsp", "parch"]

compare_panel(
    "MAR: Heatmap ordenado por pclass (padrao visivel)",
    md_original.heatmap(
        selected_columns=cols_mar,
        order_by=[{"axis": "rows", "column": "pclass", "type": "numeric", "ascending": True}],
        title="Original (sem missing)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.heatmap(
        selected_columns=cols_mar,
        order_by=[{"axis": "rows", "column": "pclass", "type": "numeric", "ascending": True}],
        title="MAR em fare (dependente de pclass)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MAR - Visualizacao 4 (ajuda a identificar)

No scatterplot, os pontos em falta de `fare` aparecem mais em certos valores de `pclass`.

In [ ]:
compare_panel(
    "MAR: Scatterplot pclass vs fare",
    md_original.scatterplot(
        x="pclass",
        y="fare",
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.scatterplot(
        x="pclass",
        y="fare",
        title="Com MAR (fare depende de pclass)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MAR - Visualizacao 5 (ajuda a identificar)

Em parallel coordinates, as linhas vermelhas (missing em `fare`) seguem um padrao
associado ao eixo de `pclass`.

In [ ]:
compare_panel(
    "MAR: Parallel coordinates (missingness ligado a pclass)",
    md_original.parallel_coordinates(
        selected_columns=["pclass", "fare", "age", "sibsp"],
        missingness_color_column="fare",
        normalize=False,
        impute_below_range_frac=0.1,
        show_colorbar=False,
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.parallel_coordinates(
        selected_columns=["pclass", "fare", "age", "sibsp"],
        missingness_color_column="fare",
        normalize=False,
        impute_below_range_frac=0.1,
        show_colorbar=False,
        title="Com MAR (fare)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

# 6. MNAR - Missing Not At Random

**Objetivo**: 30% de missingness em `fare_per_person`, dependendo do proprio valor.

O que esperamos ver:
- Missingness concentrada em valores extremos da propria variavel
- Visualmente, so conseguimos **pistas indiretas** (MNAR e o caso mais perigoso)

Em cada visualizacao, a esquerda esta o dataset **original**, e a direita o dataset **com MNAR**.

In [ ]:
MNAR_TARGET = "fare_per_person"

df_mnar = ampute_with_pyampute(
    df_complete,
    target_col=MNAR_TARGET,
    mechanism="MNAR",
    prop=0.30,
    seed=13,
)

md_mnar = MissingData(df_mnar)

### MNAR - Visualizacao 1 (nao diz o mecanismo)

In [ ]:
compare_panel(
    "MNAR: Missing count por coluna (inconclusivo)",
    md_original.missing_count_barchart(
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mnar.missing_count_barchart(
        title="Com MNAR (30% em fare_per_person)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MNAR - Visualizacao 2 (nao diz o mecanismo)

In [ ]:
compare_panel(
    "MNAR: Missing rate por coluna (inconclusivo)",
    md_original.column_missing_rate_heatmap(
        title="Original (completo)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
    md_mnar.column_missing_rate_heatmap(
        title="Com MNAR (30% em fare_per_person)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
)

### MNAR - Visualizacao 3 (ajuda a identificar)

No scatterplot, os pontos em falta de `fare_per_person` tendem a aparecer
em zonas de `fare` mais alto. Isto e um **sinal**, mas nao prova MNAR.

In [ ]:
compare_panel(
    "MNAR: Scatterplot fare_per_person vs fare",
    md_original.scatterplot(
        x="fare_per_person",
        y="fare",
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mnar.scatterplot(
        x="fare_per_person",
        y="fare",
        title="Com MNAR (fare_per_person)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MNAR - Visualizacao 4 (ajuda a identificar)

Ordenando por `fare`, vemos que a missingness em `fare_per_person` aparece
mais em tarifas altas (proxy). E um indicio, nao uma prova definitiva.

In [ ]:
cols_mnar = ["fare", "fare_per_person", "pclass", "age"]

compare_panel(
    "MNAR: Heatmap ordenado por fare (proxy)",
    md_original.heatmap(
        selected_columns=cols_mnar,
        order_by=[{"axis": "rows", "column": "fare", "type": "numeric", "ascending": False}],
        title="Original (sem missing)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mnar.heatmap(
        selected_columns=cols_mnar,
        order_by=[{"axis": "rows", "column": "fare", "type": "numeric", "ascending": False}],
        title="MNAR em fare_per_person",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

### MNAR - Visualizacao 5 (ajuda a identificar)

Em parallel coordinates, as linhas vermelhas (missing em `fare_per_person`) se alinham
com valores altos de `fare`. Mais uma pista visual do MNAR.

In [ ]:
compare_panel(
    "MNAR: Parallel coordinates (pistas via proxy)",
    md_original.parallel_coordinates(
        selected_columns=["fare_per_person", "fare", "pclass", "age"],
        missingness_color_column="fare_per_person",
        normalize=False,
        impute_below_range_frac=0.1,
        show_colorbar=False,
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mnar.parallel_coordinates(
        selected_columns=["fare_per_person", "fare", "pclass", "age"],
        missingness_color_column="fare_per_person",
        normalize=False,
        impute_below_range_frac=0.1,
        show_colorbar=False,
        title="Com MNAR (fare_per_person)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

## 7. Fecho rapido

- MCAR: padroes dispersos e sem relacao visivel
- MAR: missingness concentra-se quando ordenamos por uma variavel observada
- MNAR: apenas pistas indiretas; o mecanismo e o mais dificil de confirmar

Se quiseres, podemos ajustar as variaveis alvo, as funcoes de probabilidade
ou adicionar outros graficos do package.